# Tract scalar overlays

Port of `packages/niivue/examples/tract.tsf.html`. Loads a TCK tract with per-vertex TSF and per-streamline TXT scalar layers.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"
MESHES = f"{BASE_URL}/meshes"

dpv_name = "mni152.SLF1_R"
dps_name = "mni152.SLF1_R"

nv = NiiVue(background_color=[0, 0, 0, 1], slice_type=4, backend="webgl2")

slice_type = widgets.Dropdown(options=[("Render", 4), ("A+C+S+R", 3)], value=4, description="Slice")
fiber_color = widgets.Dropdown(
    options=[("Per-vertex TSF", "dpv"), ("Per-streamline TXT", "dps"), ("Local direction", "local"), ("Global direction", "global"), ("Fixed", "fixed")],
    value="dpv",
    description="Color",
)
fiber_colormap = widgets.Dropdown(options=["inferno", "actc", "plasma", "warm", "winter"], value="inferno", description="Colormap")
cal_min = widgets.FloatSlider(value=25.0, min=0.0, max=75.0, step=1.0, description="Min")
radius = widgets.IntSlider(value=5, min=0, max=20, step=1, description="Radius")

def update_slice(change):
    nv.slice_type = change["new"]

def update_radius(change):
    nv.set_tract_options(0, {"fiberRadius": change["new"] * 0.1})

def update_color(change=None):
    mode = fiber_color.value
    if mode == "dpv":
        nv.set_tract_options(0, {"colorBy": f"dpv:{dpv_name}", "colormap": fiber_colormap.value})
    elif mode == "dps":
        nv.set_tract_options(0, {"colorBy": f"dps:{dps_name}", "colormap": fiber_colormap.value})
    else:
        nv.set_tract_options(0, {"colorBy": mode})

def update_colormap(change):
    nv.set_tract_options(0, {"colormap": change["new"]})

def update_cal_min(change):
    nv.set_tract_options(0, {"calMin": change["new"]})

slice_type.observe(update_slice, names="value")
radius.observe(update_radius, names="value")
fiber_color.observe(update_color, names="value")
fiber_colormap.observe(update_colormap, names="value")
cal_min.observe(update_cal_min, names="value")

controls = widgets.VBox([
    widgets.HBox([slice_type, fiber_color, fiber_colormap]),
    widgets.HBox([cal_min, radius]),
])
display(controls)
display(nv)

nv.set_clip_planes([[0.1, 180, 20], [0.1, 0, -20]])
nv.load_volumes([{"url": f"{VOLUMES}/mni152.nii.gz"}])
nv.load_meshes([
    {
        "url": f"{MESHES}/tract.SLF1_R.tck",
        "rgba255": [0, 255, 255, 255],
        "layers": [
            {"url": f"{MESHES}/mni152.SLF1_R.tsf"},
            {"url": f"{MESHES}/mni152.SLF1_R.txt"},
        ],
        "tractOptions": {"colorBy": f"dpv:{dpv_name}", "colormap": "inferno"},
    }
])
update_radius({"new": radius.value})
update_cal_min({"new": cal_min.value})
